## ChessFormer Experiment

This notebook is originally executed on Colab platform. You should copy the notebook onto Colab and run it there.

Otherwise, some of the graphic interfaces can not be displayed properly.

## Installation

In [ ]:
!pip install -q chess

In [ ]:
!git clone https://huggingface.co/kaupane/ChessFormer-SL

Cloning into 'ChessFormer-SL'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 53 (delta 16), reused 0 (delta 0), pack-reused 8 (from 1)
Unpacking objects: 100% (53/53), 25.36 KiB | 1.69 MiB/s, done.


In [ ]:
%cd ChessFormer-SL

/content/ChessFormer-SL


In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## Test Transformer Model

In [ ]:
import torch
from model import ChessFormerModel

# Load model
model = ChessFormerModel.from_pretrained("kaupane/ChessFormer-SL")
model.to(device)
model.eval()

# Analyze position
fens = ["rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1"]
repetitions = torch.tensor([1]).to(device)

with torch.no_grad():
    move_logits, position_value = model(fens, repetitions)

# Get best move (requires additional processing for legal moves)
print(f"Position value: {position_value.item():.3f}")

config.json:   0%|          | 0.00/153 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/403M [00:00<?, ?B/s]

Position value: -0.062


## Agent Functions

In [ ]:
import chess
import random

def get_random_move(fen: str) -> str:
    board = chess.Board(fen)
    legal_moves = list(board.legal_moves)
    random_move = random.choice(legal_moves)
    return str(random_move)

In [ ]:
import torch
from model import ChessFormerModel
from mapping import IDX_TO_UCI_MOVE

model = ChessFormerModel.from_pretrained("kaupane/ChessFormer-SL")
model.to(device)
model.eval()

def get_smart_move(fen: str) -> str:
    fens = [fen]
    reps = torch.tensor([1]).to(device)
    with torch.inference_mode():
        move_logits, pos_value = model(fens, reps)
    move_idx = torch.argmax(move_logits)
    move = IDX_TO_UCI_MOVE[int(move_idx)]

    board = chess.Board(fen)
    if chess.Move.from_uci(move) not in board.legal_moves:
        return get_random_move(fen)
    return move

## Interactive Chess

In [ ]:
import chess
import chess.svg
import ipywidgets as widgets
from IPython.display import display, clear_output, SVG

def play_vs_agent(agent_func, human_color=chess.WHITE):
    board = chess.Board()

    board_output = widgets.Output()      # Area to display the board
    text_input = widgets.Text(           # The input box
        placeholder='Type move (e.g. e2e4, Nf3) and press Enter',
        description='<b>Your Move:</b>',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
    status_output = widgets.Output()     # Area for game status/errors

    def render_board():
        with board_output:
            clear_output(wait=True)
            is_flipped = (human_color == chess.BLACK)
            board_svg = chess.svg.board(board, size=400, flipped=is_flipped)
            display(SVG(board_svg))

    def on_human_move(change):
        user_input = change.value.strip()
        text_input.value = ""  # Clear the box for next turn

        if not user_input or board.is_game_over():
            return

        # 1. HUMAN TURN
        with status_output:
            clear_output()
            try:
                try:
                    move = chess.Move.from_uci(user_input)
                    if move not in board.legal_moves:
                        raise ValueError
                except ValueError:
                    move = board.parse_san(user_input)

                board.push(move)
                render_board()

            except ValueError:
                print(f"Invalid move: {user_input}")
                return # Stop here, let human try again

        # Check Game Over after Human Move
        if board.is_game_over():
            with status_output:
                print(f"Game Over! Result: {board.result()}")
            text_input.disabled = True
            return

        with status_output:
            print("AI is thinking...")

        try:
            # Get FEN and call your function
            fen = board.fen()
            ai_move_str = agent_func(fen)

            # Apply AI Move
            ai_move = chess.Move.from_uci(ai_move_str)
            if ai_move in board.legal_moves:
                board.push(ai_move)
                render_board()
                with status_output:
                    print(f"AI played: {ai_move_str}")
            else:
                with status_output:
                    print(f"AI attempted illegal move: {ai_move_str}")
        except Exception as e:
            with status_output:
                print(f"Error in AI function: {e}")

        # Check Game Over after AI Move
        if board.is_game_over():
            with status_output:
                print(f"Game Over! Result: {board.result()}")
            text_input.disabled = True

    text_input.on_submit(on_human_move)

    display(board_output)
    display(status_output)
    display(text_input)

    # Initial render
    render_board()

    # If AI is White (plays first), trigger one AI move immediately
    if human_color == chess.BLACK:
        # Manually trigger AI turn logic (simplified for this snippet)
        with status_output: print("AI is thinking (First move)...")
        ai_move_str = agent_func(board.fen())
        board.push(chess.Move.from_uci(ai_move_str))
        render_board()
        with status_output: clear_output()

In [ ]:
play_vs_agent(agent_func=get_smart_move)

Output()

Output()

Text(value='', description='<b>Your Move:</b>', layout=Layout(width='500px'), placeholder='Type move (e.g. e2e…

## Evaluation

In [ ]:
import chess
from tqdm.notebook import tqdm

def play_match(white_agent, black_agent, max_moves=200):
    board = chess.Board()
    moves_count = 0

    while not board.is_game_over() and moves_count < max_moves:
        fen = board.fen()

        try:
            if board.turn == chess.WHITE:
                move_str = white_agent(fen)
            else:
                move_str = black_agent(fen)

            move = chess.Move.from_uci(move_str)

            if move in board.legal_moves:
                board.push(move)
            else:
                # If an agent makes an illegal move, they instantly lose
                print(f"Illegal move by {'White' if board.turn == chess.WHITE else 'Black'}: {move_str}")
                return 0.0 if board.turn == chess.WHITE else 1.0

        except Exception as e:
            # If an agent crashes, they lose
            return 0.0 if board.turn == chess.WHITE else 1.0

        moves_count += 1

    result = board.result()

    if result == '1-0':
        return 1.0
    elif result == '0-1':
        return 0.0
    elif result == '1/2-1/2':
        return 0.5
    else:
        # Game hit max_moves without a clear result -> Treat as Draw
        return 0.5

def evaluate_agents(smart_agent, random_agent, num_games=100):
    """
    Plays num_games between smart_agent and random_agent.
    Swaps colors halfway through.
    """
    if num_games % 2 != 0:
        print("Warning: num_games should be even to ensure equal color distribution. Adding 1 game.")
        num_games += 1

    smart_wins = 0
    random_wins = 0
    draws = 0

    print(f"Starting evaluation: {num_games} games...")

    for i in tqdm(range(num_games)):
        # First half: Smart Agent plays WHITE
        if i < num_games // 2:
            score = play_match(white_agent=smart_agent, black_agent=random_agent)
            if score == 1.0:
                smart_wins += 1
            elif score == 0.0:
                random_wins += 1
            else:
                draws += 1

        # Second half: Smart Agent plays BLACK
        else:
            score = play_match(white_agent=random_agent, black_agent=smart_agent)
            # Note: score is 1.0 if White (Random) wins, 0.0 if Black (Smart) wins
            if score == 1.0:
                random_wins += 1
            elif score == 0.0:
                smart_wins += 1
            else:
                draws += 1

    # --- Statistics ---
    win_rate = (smart_wins / num_games) * 100
    non_loss_rate = ((smart_wins + draws) / num_games) * 100

    print("\n" + "="*30)
    print(f"EVALUATION RESULTS ({num_games} Games)")
    print("="*30)
    print(f"Smart Agent Wins: {smart_wins}")
    print(f"Random Agent Wins: {random_wins}")
    print(f"Draws:            {draws}")
    print("-" * 30)
    print(f"Smart Agent Win Rate:      {win_rate:.2f}%")
    print(f"Smart Agent Non-Loss Rate: {non_loss_rate:.2f}%")

In [ ]:
evaluate_agents(get_smart_move, get_random_move, num_games=1000)

Starting evaluation: 1000 games...


  0%|          | 0/1000 [00:00<?, ?it/s]


EVALUATION RESULTS (1000 Games)
Smart Agent Wins: 943
Random Agent Wins: 0
Draws:            57
------------------------------
Smart Agent Win Rate:      94.30%
Smart Agent Non-Loss Rate: 100.00%
